# Agent Memory with Redis Cloud and Snowflake Cortex – v5 (improved)

This notebook builds a travel agent with:

- **Snowflake Cortex** for LLM chat and embeddings
- **Redis Cloud** for short‑term checkpoints and long‑term vector memory
- **LangGraph** for the agent workflow
- **Conversation persistence** (resume any previous session)
- **Tool‑call loop prevention** (hard cap of 3 tool calls per turn)
- **In‑chat commands** (`exit`, `debug`, `history`, `memories`)

All API calls stay inside Snowflake (no OpenAI).

## 1. Install Dependencies

In [ ]:
%pip install langgraph langgraph-checkpoint redis redisvl ulid pydantic requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Load Environment Variables

In [13]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}:")

_set_env("SNOWFLAKE_ACCOUNT")
_set_env("SNOWFLAKE_PAT")
_set_env("MODEL")
_set_env("REDIS_URL")   # e.g., rediss://:password@host:port

## 3. Connect to Redis Cloud

In [14]:
from redis import Redis

REDIS_URL = os.environ["REDIS_URL"]
redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()  # should return True
print("✅ Connected to Redis")

✅ Connected to Redis


## 4. Define Memory Data Models (Pydantic)

In [15]:
import ulid
from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

class MemoryType(str, Enum):
    EPISODIC = "episodic"
    SEMANTIC = "semantic"

class Memory(BaseModel):
    content: str
    memory_type: MemoryType
    metadata: str

class StoredMemory(Memory):
    id: str
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None

print("✅ Data models ready")

✅ Data models ready


## 5. Create RedisVL Vector Index for Long‑Term Memory

In [16]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema

VECTOR_DIM = 1024  # snowflake-arctic-embed-l-v2.0

memory_schema = IndexSchema.from_dict({
    "index": {
        "name": "agent_memories",
        "prefix": "memory",
        "key_separator": ":",
        "storage_type": "json",
    },
    "fields": [
        {"name": "content", "type": "text"},
        {"name": "memory_type", "type": "tag"},
        {"name": "metadata", "type": "text"},
        {"name": "created_at", "type": "text"},
        {"name": "user_id", "type": "tag"},
        {"name": "memory_id", "type": "tag"},
        {"name": "thread_id", "type": "tag"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "flat",
                "dims": VECTOR_DIM,
                "distance_metric": "cosine",
                "datatype": "float32",
            },
        },
    ],
})

long_term_memory_index = SearchIndex(
    schema=memory_schema,
    redis_client=redis_client,
    validate_on_load=True
)
long_term_memory_index.create(overwrite=True)
print("✅ Long-term memory index ready")

✅ Long-term memory index ready


## 6. Snowflake Embedding Helper

In [17]:
import requests
import json

ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
PAT = os.environ["SNOWFLAKE_PAT"]
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"

def embed_text(text: str) -> List[float]:
    """Call Snowflake Cortex Embed API."""
    url = f"https://{ACCOUNT}.snowflakecomputing.com/api/v2/cortex/inference:embed"
    headers = {
        "Authorization": f"Bearer {PAT}",
        "Content-Type": "application/json",
    }
    payload = {"model": EMBED_MODEL, "input": text}
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Snowflake Embed API error {response.status_code}: {response.text}")
    return response.json()["data"][0]["embedding"]

# Quick test
# print(embed_text("hello")[:5])

## 7. Memory Operations (Store, Retrieve, Deduplicate)

In [18]:
import logging
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag

logger = logging.getLogger(__name__)
SYSTEM_USER_ID = "system"

def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    embedding = embed_text(content)
    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type.value)
    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)
    q = VectorRangeQuery(
        vector=embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    return len(long_term_memory_index.query(q)) > 0

def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
) -> None:
    if metadata is None:
        metadata = "{}"
    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory exists — skipping.")
        return
    embedding = embed_text(content)
    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id or "",
    }
    long_term_memory_index.load([memory_data])
    logger.info(f"💾 Stored [{memory_type.value}] memory: {content!r}")

def retrieve_memories(
    query: str,
    memory_type: Optional[MemoryType] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.3,
    limit: int = 5,
) -> List[StoredMemory]:
    embedding = embed_text(query)
    filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]
    if memory_type:
        filters.append(f"@memory_type:{{{memory_type.value}}}")
    if thread_id:
        filters.append(f"@thread_id:{{{thread_id}}}")
    q = VectorRangeQuery(
        vector=embedding,
        return_fields=["content", "memory_type", "metadata", "created_at", "memory_id", "thread_id", "user_id"],
        num_results=limit,
        vector_field_name="embedding",
        distance_threshold=distance_threshold,
        dialect=2,
    )
    q.set_filter(" ".join(filters))
    results = long_term_memory_index.query(q)
    memories = []
    for doc in results:
        try:
            memories.append(StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id") or None,
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            ))
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
    return memories

print("✅ Memory operations ready")

✅ Memory operations ready


## 8. Define Tools (Store & Retrieve) for the Agent

In [19]:
def store_memory_tool(
    content: str,
    memory_type: str,
    metadata: Optional[dict] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type)
        store_memory(content, mem_type, user_id, thread_id, str(metadata) if metadata else None)
        return f"✅ Stored [{mem_type.value}] memory: {content}"
    except Exception as e:
        return f"❌ Error storing memory: {e}"

def retrieve_memories_tool(
    query: str,
    memory_type: Optional[str] = None,
    limit: int = 5,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type) if memory_type else None
        memories = retrieve_memories(query, mem_type, user_id, thread_id, limit=limit)
        if not memories:
            return "No relevant memories found."
        lines = ["🧠 Long-term memories:"]
        for m in memories:
            lines.append(f"  - [{m.memory_type.value}] {m.content}")
        return "\n".join(lines)
    except Exception as e:
        return f"❌ Error retrieving memories: {e}"

# Tool registry (used by the graph)
TOOLS = [
    {
        "name": "store_memory",
        "func": store_memory_tool,
        "description": "Store a long-term memory. Parameters: content (str), memory_type ('episodic' or 'semantic'), metadata (optional dict).",
    },
    {
        "name": "retrieve_memories",
        "func": retrieve_memories_tool,
        "description": "Retrieve relevant memories via semantic search. Parameters: query (str), memory_type (optional), limit (int).",
    },
]

print("✅ Tools ready")

✅ Tools ready


## 9. Snowflake Cortex Chat LLM (with Tool‑Calling via Prompt Engineering)

In [36]:
import re
import json
import ulid
import requests
import logging
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks import CallbackManagerForLLMRun
from typing import Any, List, Optional, Sequence

logger = logging.getLogger(__name__)

def extract_tool_call(text: str) -> Optional[dict]:
    """Robustly extract {tool, arguments} JSON from any text."""
    # Remove markdown code fences
    cleaned = re.sub(r"```(?:json)?\n?|```", "", text).strip()
    # Find the outermost JSON object
    start = cleaned.find('{')
    if start == -1:
        start = cleaned.find('{"tool"')
    if start != -1:
        depth = 0
        for i, ch in enumerate(cleaned[start:], start=start):
            if ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try:
                        candidate = cleaned[start:i+1]
                        obj = json.loads(candidate)
                        if "tool" in obj and "arguments" in obj:
                            return obj
                    except json.JSONDecodeError:
                        pass
                    break
    return None

class SnowflakeCortexLLM(BaseChatModel):
    account: str
    pat: str
    model: str
    temperature: float = 0.7

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        # Convert to Snowflake API format
        api_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                api_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                api_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                api_messages.append({"role": "assistant", "content": msg.content})
            elif isinstance(msg, ToolMessage):
                api_messages.append({
                    "role": "user",
                    "content": f"[Tool result for '{msg.name}']:\n{msg.content}"
                })

        url = f"https://{self.account}.snowflakecomputing.com/api/v2/cortex/v1/chat/completions"
        headers = {"Authorization": f"Bearer {self.pat}", "Content-Type": "application/json"}
        payload = {
            "model": self.model,
            "messages": api_messages,
            "temperature": self.temperature,
        }
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code != 200:
            raise RuntimeError(f"Snowflake API error {response.status_code}: {response.text}")

        data = response.json()
        raw_content = data["choices"][0]["message"]["content"].strip()
        logger.debug(f"Raw LLM response: {raw_content!r}")

        # ── Fallback if response is malformed ──────────────────────────────
        if raw_content in ("}", "", "{}"):
            logger.warning("Malformed response – retrying with plain‑text fallback.")
            # Force a simple greeting
            fallback_payload = {
                "model": self.model,
                "messages": [{"role": "user", "content": "Reply with a friendly greeting."}],
                "temperature": self.temperature,
            }
            fallback_resp = requests.post(url, headers=headers, json=fallback_payload)
            if fallback_resp.status_code == 200:
                raw_content = fallback_resp.json()["choices"][0]["message"]["content"].strip()
            else:
                raw_content = "Hello! How can I help you?"

        # ── Parse tool call ─────────────────────────────────────────────────
        tool_call = extract_tool_call(raw_content)
        clean_content = raw_content
        if tool_call:
            # Remove the JSON block from the visible content
            clean_content = re.sub(r"\{[\"']tool[\"'][\s\S]*?\}", "", raw_content).strip()
            if not clean_content:
                clean_content = "Let me use a tool to help you."

        ai_message = AIMessage(
            content=clean_content,
            tool_calls=[{
                "name": tool_call["tool"],
                "args": tool_call["arguments"],
                "id": f"call_{ulid.ULID()}",
            }] if tool_call else []
        )
        return ChatResult(generations=[ChatGeneration(message=ai_message)])

    @property
    def _llm_type(self) -> str:
        return "snowflake-cortex"

# Initialize
account = os.environ["SNOWFLAKE_ACCOUNT"]
pat = os.environ["SNOWFLAKE_PAT"]
model = os.environ["MODEL"]
llm = SnowflakeCortexLLM(account=account, pat=pat, model=model, temperature=0.7)
print("✅ Snowflake Cortex LLM ready")

✅ Snowflake Cortex LLM ready


## 10. Conversation Persistence (Redis)

In [37]:
CONV_PREFIX = "conversation"

def save_conversation_to_redis(state, thread_id: str, user_id: str) -> None:
    transcript = []
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            transcript.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                transcript.append({"role": "assistant", "content": content})
        elif isinstance(m, ToolMessage):
            transcript.append({"role": "tool", "name": m.name, "content": m.content})
        elif isinstance(m, SystemMessage):
            transcript.append({"role": "summary", "content": m.content})
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    redis_client.set(key, json.dumps(transcript))
    logger.info(f"💾 Conversation saved → {key} ({len(transcript)} turns)")

def load_conversation_from_redis(thread_id: str, user_id: str) -> list:
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    raw = redis_client.get(key)
    return json.loads(raw) if raw else []

def rebuild_state_from_transcript(transcript: list):
    messages = []
    for turn in transcript:
        role = turn.get("role")
        content = turn.get("content", "")
        if role == "user":
            messages.append(HumanMessage(content=content))
        elif role == "assistant":
            messages.append(AIMessage(content=content))
        elif role == "summary":
            messages.append(SystemMessage(content=content))
    return messages

def get_all_conversation_keys(user_id: str) -> list:
    pattern = f"{CONV_PREFIX}:{user_id}:*"
    keys = [k.decode() for k in redis_client.keys(pattern)]
    return sorted(keys)

def select_conversation(user_id: str = "demo_user") -> tuple:
    """Interactive selector; returns (thread_id, transcript)."""
    keys = get_all_conversation_keys(user_id)

    print("\n" + "=" * 55)
    print("  📚 CONVERSATION SELECTOR")
    print("=" * 55)

    if not keys:
        print("  No previous conversations found.")
        print("  Starting a new conversation.\n")
        thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        return thread_id, []

    print("  [0] 🆕 Start a new conversation")
    print()
    for i, key in enumerate(keys, start=1):
        thread_part = key.split(":", 2)[2] if key.count(":") >= 2 else key
        raw = redis_client.get(key)
        turns = len(json.loads(raw)) if raw else 0
        preview = ""
        if raw:
            transcript = json.loads(raw)
            for turn in transcript:
                if turn.get("role") == "user":
                    preview = turn.get("content", "")[:60]
                    break
        print(f"  [{i}] 🗂  {thread_part}")
        print(f"       {turns} turns  |  \"{preview}{'...' if len(preview) == 60 else ''}\"")
        print()

    print("=" * 55)

    while True:
        try:
            choice = input(f"  Choose [0-{len(keys)}]: ").strip()
        except (EOFError, KeyboardInterrupt):
            choice = "0"

        if choice == "0":
            thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            print(f"\n  🆕 New conversation started: {thread_id}\n")
            return thread_id, []

        if choice.isdigit() and 1 <= int(choice) <= len(keys):
            idx = int(choice) - 1
            chosen_key = keys[idx]
            thread_id = chosen_key.split(":", 2)[2]
            transcript = json.loads(redis_client.get(chosen_key))
            print(f"\n  📂 Resuming: {thread_id} ({len(transcript)} turns)\n")
            return thread_id, transcript

        print(f"  ❌ Invalid choice. Enter a number between 0 and {len(keys)}.")

def print_conversation_from_redis(thread_id: str, user_id: str = "demo_user") -> None:
    transcript = load_conversation_from_redis(thread_id, user_id)
    if not transcript:
        print(f"No conversation found for user={user_id!r} thread={thread_id!r}")
        return
    print(f"\n{'='*60}")
    print(f"CONVERSATION  user={user_id}  thread={thread_id}")
    print(f"{'='*60}")
    for turn in transcript:
        role = turn["role"].upper()
        name = f" ({turn['name']})" if turn.get("name") else ""
        print(f"\n[{role}{name}]\n{turn.get('content', '')}")
    print()

print("✅ Conversation persistence helpers ready")

✅ Conversation persistence helpers ready


## 11. LangGraph Workflow (with tool‑call cap)

In [38]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables.config import RunnableConfig
from typing import Annotated

# ── State with tool‑call counter ──────────────────────────────────────────────
class RuntimeState(MessagesState):
    tool_call_count: int   # reset to 0 at start of each user turn

MAX_TOOL_CALLS_PER_TURN = 3

SYSTEM_PROMPT = """You are a travel assistant with persistent memory.

You have two tools:
1. store_memory(content, memory_type, metadata)
   Save facts about the user.
   memory_type: "episodic" for personal preferences/trips/facts, "semantic" for general knowledge.

2. retrieve_memories(query, memory_type, limit)
   Search past memories by semantic similarity.

YOUR RULES — follow these every turn in order:

STEP 1 — Call retrieve_memories FIRST.
{"tool": "retrieve_memories", "arguments": {"query": "user travel preferences and history", "memory_type": "episodic", "limit": 5}}

STEP 2 — After the tool result appears, call store_memory if the user revealed:
  - A destination they plan to visit
  - A travel preference (budget, solo, family, interests, etc.)
  - Their home city or travel dates
{"tool": "store_memory", "arguments": {"content": "...", "memory_type": "episodic"}}

STEP 3 — Write your reply in plain text. No more tool calls after this.

IMPORTANT: After you see a [Tool result] message, you MUST either:
  a) Make ONE more tool call (store_memory only), then write your reply, OR
  b) Write your reply immediately.
Never call retrieve_memories more than once per turn.
"""

# ── Node 1: Agent ──────────────────────────────────────────────────────────────

def respond_to_user(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    user_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not user_msgs:
        return state

    tool_calls_this_turn = state.get("tool_call_count", 0)

    # Build BaseMessage list for the LLM
    llm_messages = []
    # Insert system prompt at the beginning
    llm_messages.append(SystemMessage(content=SYSTEM_PROMPT))

    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            llm_messages.append(HumanMessage(content=m.content))
        elif isinstance(m, AIMessage):
            # Skip messages that only contained tool calls (they have no useful content)
            if m.content and not extract_tool_call(m.content):
                llm_messages.append(AIMessage(content=m.content))
        elif isinstance(m, ToolMessage):
            # Convert tool results to user messages so the model sees them
            llm_messages.append(HumanMessage(
                content=f"[Tool result for '{m.name}']:\n{m.content}"
            ))
        elif isinstance(m, SystemMessage):
            # Already inserted system prompt; skip others (like summaries)
            pass

    # If we hit the cap, force plain reply
    if tool_calls_this_turn >= MAX_TOOL_CALLS_PER_TURN:
        llm_messages.append(HumanMessage(
            content=(
                "You have already used the maximum number of tool calls for this turn. "
                "Do NOT make any more tool calls. Write your final reply to the user now in plain text."
            )
        ))

    # Invoke the LLM (returns an AIMessage with tool_calls already parsed)
    ai_msg = llm.invoke(llm_messages)
    state["messages"].append(ai_msg)
    return state

# ── Node 2: Execute Tools ──────────────────────────────────────────────────────

def execute_tools(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    ai_msgs = [
        m for m in state["messages"]
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None)
    ]
    if not ai_msgs:
        return state

    latest = ai_msgs[-1]
    configurable = config.get("configurable", {}) if config else {}
    user_id = configurable.get("user_id", SYSTEM_USER_ID)
    thread_id = configurable.get("thread_id")

    for tc in latest.tool_calls:
        tool_name = tc["name"]
        tool_args = dict(tc["args"])
        tool_func = next((t["func"] for t in TOOLS if t["name"] == tool_name), None)

        if not tool_func:
            result = f"Tool '{tool_name}' not found."
        else:
            tool_args.setdefault("user_id", user_id)
            tool_args.setdefault("thread_id", thread_id)
            try:
                result = tool_func(**tool_args)
                logger.info(f"Tool '{tool_name}' result: {result!r}")
            except Exception as e:
                result = f"Error executing tool '{tool_name}': {e}"
                logger.error(result)

        state["messages"].append(ToolMessage(
            content=str(result),
            tool_call_id=tc["id"],
            name=tool_name,
        ))

    state["tool_call_count"] = state.get("tool_call_count", 0) + 1
    return state

# ── Node 3: Summarise ──────────────────────────────────────────────────────────

MESSAGE_THRESHOLD = 8

def summarize_conversation(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    messages = state["messages"]
    if len(messages) < MESSAGE_THRESHOLD:
        return state

    readable = []
    for m in messages:
        if isinstance(m, HumanMessage):
            readable.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                readable.append(f"Assistant: {content}")

    if not readable:
        return state

    summary_prompt = (
        "Summarise this travel assistant conversation in 3-5 sentences. "
        "Focus on destinations, user preferences, and decisions made.\n\n"
        + "\n".join(readable)
    )
    # Use the LLM directly for summarisation (no tools needed)
    summary_msg = llm.invoke([
        SystemMessage(content="You are a concise conversation summariser."),
        HumanMessage(content=summary_prompt)
    ])
    logger.info(f"Summarised {len(messages)} messages.")

    summary_sys = SystemMessage(
        content=f"Summary of conversation so far:\n\n{summary_msg.content}\n\nContinue from here."
    )
    keep = messages[-2:] if len(messages) >= 2 else messages
    state["messages"] = [summary_sys] + keep
    return state

# ── Routing ────────────────────────────────────────────────────────────────────

def decide_next(state: RuntimeState) -> str:
    last = state["messages"][-1] if state["messages"] else None
    if state.get("tool_call_count", 0) >= MAX_TOOL_CALLS_PER_TURN:
        return "summarize"
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "execute_tools"
    return "summarize"

# ── Compile Graph ──────────────────────────────────────────────────────────────

workflow = StateGraph(RuntimeState)
workflow.add_node("agent", respond_to_user)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("summarize", summarize_conversation)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    decide_next,
    {"execute_tools": "execute_tools", "summarize": "summarize"},
)
workflow.add_edge("execute_tools", "agent")
workflow.add_edge("summarize", END)

# Use in‑memory checkpointer (conversation is persisted manually)
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("✅ Graph compiled")

✅ Graph compiled


## 12. Run the Agent (Interactive Loop)

In [39]:
def main(user_id: str = "demo_user", verbose: bool = False):

    # ── Conversation selector ─────────────────────────────────────────────────
    thread_id, previous_transcript = select_conversation(user_id=user_id)

    print(f"🌍 Travel Assistant with Memory (Snowflake Cortex)")
    print(f"   user={user_id}  thread={thread_id}")
    print("   Commands: exit | quit | debug | history | memories\n")

    config = {
        "configurable": {"thread_id": thread_id, "user_id": user_id},
        "recursion_limit": 50,
    }

    # Rebuild state from saved transcript (empty list = new conversation)
    state = RuntimeState(
        messages=rebuild_state_from_transcript(previous_transcript),
        tool_call_count=0
    )

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            save_conversation_to_redis(state, thread_id, user_id)
            break

        if not user_input:
            continue

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            save_conversation_to_redis(state, thread_id, user_id)
            break

        if user_input.lower() == "debug":
            verbose = not verbose
            print(f"  [verbose mode {'ON' if verbose else 'OFF'}]\n")
            continue

        if user_input.lower() == "history":
            print_conversation_from_redis(thread_id, user_id)
            continue

        if user_input.lower() == "memories":
            keys = redis_client.keys("memory:*")
            print(f"\n🧠 Long-term memories ({len(keys)} total):")
            for key in sorted(keys):
                data = redis_client.json().get(key)
                if data:
                    ts  = data.get("created_at", "")[:19]
                    mt  = data.get("memory_type", "?")
                    cnt = data.get("content", "")
                    print(f"  [{mt:8s}] {ts}  {cnt}")
            print()
            continue

        # ── Reset tool‑call counter for each new user turn ──────────────────
        state["tool_call_count"] = 0
        state["messages"].append(HumanMessage(content=user_input))

        try:
            for result in graph.stream(state, config=config, stream_mode="values"):
                state = RuntimeState(**result)

            # Find the last clean assistant reply
            reply = None
            for m in reversed(state["messages"]):
                if isinstance(m, AIMessage):
                    content = m.content.strip() if m.content else ""
                    if content and not extract_tool_call(content):
                        reply = content
                        break

            if reply:
                print(f"\nAssistant: {reply}\n")
            else:
                print("\nAssistant: (no reply generated — try again)\n")

            save_conversation_to_redis(state, thread_id, user_id)

            if verbose:
                print(f"── DEBUG (tool_call_count={state.get('tool_call_count', 0)}) ──")
                for i, m in enumerate(state["messages"]):
                    kind = type(m).__name__
                    content = (m.content or "")[:120].replace("\n", " ")
                    tc = f" [calls={[t['name'] for t in getattr(m, 'tool_calls', [])]}]" if getattr(m, "tool_calls", None) else ""
                    print(f"  [{i:02d}] {kind}{tc}: {content!r}")
                print("──────────────────────────────────────\n")

        except Exception as e:
            logger.error(f"Error during graph execution: {e}", exc_info=True)
            print(f"\n⚠️  Error: {e}\n")

    return state

# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    user_id = input("Enter user ID (default demo_user): ") or "demo_user"
    final_state = main(user_id=user_id, verbose=False)


  📚 CONVERSATION SELECTOR
  [0] 🆕 Start a new conversation

  [1] 🗂  thread_20260806_131441
       6 turns  |  "hello"

  [2] 🗂  thread_20260806_131846
       18 turns  |  "hi"


  🆕 New conversation started: thread_20260806_134036

🌍 Travel Assistant with Memory (Snowflake Cortex)
   user=demo_user  thread=thread_20260806_134036
   Commands: exit | quit | debug | history | memories


Assistant: It seems like we don't have any information about your travel preferences or history yet. 

You're planning a trip to Hampi! That's a great choice. Hampi is a UNESCO World Heritage Site in Karnataka, India, known for its ancient ruins and historical significance. What are your interests? Are you looking for a relaxing getaway or an adventurous trip?


Assistant: It seems like there was an error storing your travel plans to Hampi.

No worries, I'll make sure to keep track of our conversation. So, to recap, you're planning a 2-day, 1-night trip to Hampi, and I've provided you with some estimates

# 13. Inspect Conversations and Memories (Optional)

In [40]:
# List all saved conversations
from redis import Redis
keys = redis_client.keys("conversation:demo_user:*")
print("Conversations:")
for k in sorted(keys):
    print(k.decode())

Conversations:
conversation:demo_user:thread_20260806_131441
conversation:demo_user:thread_20260806_131846
conversation:demo_user:thread_20260806_134036


In [43]:
# View a specific conversation
print_conversation_from_redis(thread_id="thread_20260806_134036", user_id="demo_user")


CONVERSATION  user=demo_user  thread=thread_20260806_134036

[USER]
hampi trip

[ASSISTANT]
STEP 1: 
}

[TOOL (retrieve_memories)]
❌ Error retrieving memories: Snowflake Embed API error 400: {"code":"390400","message":"The request entity had the following errors: text size must be between 1 and 2147483647 (was [])","request_id":"2f6f29e4-0320-4dd8-bad1-97bed34a7eb9","error_code":"390400"}

[ASSISTANT]
It seems like we don't have any information about your travel preferences or history yet. 

You're planning a trip to Hampi! That's a great choice. Hampi is a UNESCO World Heritage Site in Karnataka, India, known for its ancient ruins and historical significance. What are your interests? Are you looking for a relaxing getaway or an adventurous trip?

[USER]
travel to hampi 2 days, one night rental bike flight prices which is best pre historic drawing monuments and natural wonders to see budget total

[ASSISTANT]
Sounds like you're planning a quick trip to Hampi!

For a 2-day, 1-night tri

In [42]:
# List all long‑term memories (without embedding)
keys = redis_client.keys("memory:*")
print(f"Total memories: {len(keys)}")
for key in sorted(keys):
    data = redis_client.json().get(key)
    if data:
        print(json.dumps({k: v for k, v in data.items() if k != "embedding"}, indent=2))
        print("-" * 40)

Total memories: 0
